# 09 · Figuras y matriz de confusión del documento final

Genera las **Figuras 1–4** del documento final a partir de los resultados de los notebooks anteriores. No entrena modelos: solo lee archivos ya generados.

| Salida | Qué muestra | Insumo |
|---|---|---|
| Figura 1 | Acierto de la persistencia a 1–4 meses (justifica el horizonte de 2 meses) | `serie_nivel2_<cod>.csv` (notebook 02) |
| Figura 2 | Casos y umbral del canal endémico en los 4 municipios | `serie_nivel2_<cod>.csv` (notebook 02) |
| Figura 3 | Probabilidad de XGBoost en la prueba frente al exceso observado | `predicciones_test.csv` (notebook 03) |
| Figura 4 | Matriz de confusión de XGBoost en la prueba, con sus valores en tabla | `predicciones_test.csv` (notebook 03) |

**Dónde corre:** en Colab lee de `MyDrive/Datos/dengue/procesado` (misma ruta que los notebooks 02 y 03). En local, desde la carpeta `notebooks/` del repositorio, lee `data/` y `resultados/locales/`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA = Path('/content/drive/MyDrive/Datos/dengue/procesado')
    serie = lambda cod: RUTA / f'serie_nivel2_{cod}.csv'
    def pred(cod):  # el 03 de Ibagué guarda en modelos/; el 06 y el 08 en modelos_<cod>/
        p = RUTA / f'modelos_{cod}' / 'predicciones_test.csv'
        return p if p.exists() or cod != '73001' else RUTA / 'modelos' / 'predicciones_test.csv'
    SALIDA = RUTA / 'figuras_documento'
else:
    RAIZ = Path('..') if Path('../data').exists() else Path('.')
    serie = lambda cod: RAIZ / 'data' / f'serie_nivel2_{cod}.csv'
    pred = lambda cod: RAIZ / 'resultados' / 'locales' / f'predicciones_test_{cod}.csv'
    SALIDA = RAIZ / 'resultados' / 'figuras_documento'
SALIDA.mkdir(parents=True, exist_ok=True)

MUNICIPIOS = {'Ibagué': '73001', 'Villavicencio': '50001', 'Cali': '76001', 'Montería': '23001'}
# Umbral de decisión de XGBoost elegido en validación (salida 'umbral' de los notebooks 03 de cada municipio)
UMBRAL_XGB = {'Ibagué': 0.20, 'Villavicencio': 0.45, 'Cali': 0.40}

series = {m: pd.read_csv(serie(c), parse_dates=['mes']).set_index('mes') for m, c in MUNICIPIOS.items()}
preds = {m: pd.read_csv(pred(MUNICIPIOS[m]), parse_dates=['mes_objetivo']) for m in UMBRAL_XGB}
for m, d in series.items():
    print(f'{m:14s} etiquetas {d.exceso.notna().sum():3d} meses, último umbral {d.umbral.last_valid_index():%Y-%m}')

Ibagué         etiquetas 168 meses, último umbral 2025-12
Villavicencio  etiquetas 168 meses, último umbral 2025-12
Cali           etiquetas 156 meses, último umbral 2024-12
Montería       etiquetas 132 meses, último umbral 2021-12


In [2]:
# Estilo común (un color fijo por municipio en todas las figuras; decimales con coma)
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold', 'axes.edgecolor': '#9a9994',
    'axes.linewidth': 0.8, 'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.7, 'axes.axisbelow': True,
    'xtick.color': '#52514e', 'ytick.color': '#52514e', 'axes.labelcolor': '#2b2b2a',
    'legend.edgecolor': '#d6d5d0', 'legend.fontsize': 8.5, 'savefig.dpi': 220})
COL = {'Ibagué': '#2a78d6', 'Villavicencio': '#eb6834', 'Cali': '#1baf7a', 'Montería': '#c98500'}
GRIS, UMB, FONDO = '#52514e', '#3a3936', '#efeeea'
MES = ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']
coma = lambda dec: FuncFormatter(lambda v, p: f'{v:.{dec}f}'.replace('.', ','))
miles = FuncFormatter(lambda v, p: f'{v:,.0f}'.replace(',', '.'))

## Figura 1. Acierto de la persistencia según los meses de anticipación

La persistencia predice que el mes objetivo tendrá el mismo estado (exceso o no) que el último mes conocido. Si a un mes de distancia ya acierta casi siempre, un modelo aporta poco; por eso se mide cómo cae ese acierto al alejar el horizonte. Se usan todos los meses con etiqueta desde 2012.

In [3]:
H = [1, 2, 3, 4]
acierto = {}
for m, d in series.items():
    acierto[m] = [(lambda x: (x.y == x.p).mean())(pd.DataFrame({'y': d.exceso, 'p': d.exceso.shift(h)}).dropna()) for h in H]
tabla_h = pd.DataFrame(acierto, index=[f'{h} mes' + ('es' if h > 1 else '') for h in H]).T
juntos = []
for h in H:
    x = pd.concat([pd.DataFrame({'y': d.exceso, 'p': d.exceso.shift(h)}).dropna() for d in series.values()])
    juntos.append((x.y == x.p).mean())
tabla_h.loc['Los cuatro juntos'] = juntos
display(tabla_h.round(3))

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.axvspan(1.75, 2.25, color=FONDO, zorder=0, lw=0)
ax.text(2, 0.955, 'Horizonte\nelegido', ha='center', va='top', fontsize=7.5, color=GRIS)
for m, r in acierto.items():
    ax.plot(H, r, marker='o', ms=6, lw=2, color=COL[m], label=m, markeredgecolor='white', markeredgewidth=1)
    ax.annotate(m, xy=(4, r[-1]), xytext=(7, {'Montería': 5, 'Cali': -5}.get(m, 0)), textcoords='offset points',
                va='center', fontsize=8, color='#2b2b2a')
ax.set_xticks(H); ax.set_xticklabels(['1 mes', '2 meses', '3 meses', '4 meses']); ax.set_xlim(0.8, 4.75)
ax.set_ylim(0.6, 0.96); ax.yaxis.set_major_formatter(coma(2))
ax.set_xlabel('Meses de anticipación'); ax.set_ylabel('Acierto de la persistencia')
ax.legend(loc='lower left', ncol=2)
fig.tight_layout(); fig.savefig(SALIDA / 'Figura1_persistencia_horizonte.png', facecolor='white'); plt.show()

,1 mes,2 meses,3 meses,4 meses
Ibagué,0.904,0.867,0.818,0.780
Villavicencio,0.856,0.795,0.745,0.713
Cali,0.897,0.812,0.739,0.664
Montería,0.824,0.708,0.682,0.672
Los cuatro juntos,0.873,0.800,0.750,0.711


## Figura 2. Canal endémico en los cuatro municipios

Casos mensuales, umbral del canal endémico (Bortman sin años epidémicos) y meses en exceso. El fondo gris marca el periodo de prueba. Montería no tiene umbral desde 2022, así que no tiene prueba.

In [4]:
fig, axs = plt.subplots(2, 2, figsize=(7.2, 5.0), sharex=True)
for ax, (m, d0) in zip(axs.flat, series.items()):
    d = d0.loc['2012':]
    if m != 'Montería':
        ax.axvspan(pd.Timestamp('2023-01-01'), d.umbral.last_valid_index() + pd.offsets.MonthEnd(1), color=FONDO, zorder=0, lw=0)
    ax.plot(d.index, d.casos, color=COL[m], lw=1.4, zorder=3)
    ax.plot(d.index, d.umbral, color=UMB, lw=1.1, ls=(0, (4, 2)), zorder=4)
    ex = d.exceso.fillna(0).astype(bool)
    ax.fill_between(d.index, d.umbral, d.casos, where=ex & (d.casos >= d.umbral), color=COL[m], alpha=0.28, lw=0, zorder=2, interpolate=True)
    ax.set_title(m, loc='left'); ax.yaxis.set_major_formatter(miles); ax.set_ylim(bottom=0)
    if m == 'Montería':
        ax.text(0.03, 0.95, 'Etiquetas solo hasta dic 2021:\nsin umbral desde 2022 ni periodo de prueba',
                transform=ax.transAxes, fontsize=7.3, color=GRIS, ha='left', va='top')
    ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for ax in axs[:, 0]: ax.set_ylabel('Casos por mes')
leyenda = [Line2D([], [], color='#6d6c68', lw=1.4, label='Casos notificados'),
           Line2D([], [], color=UMB, lw=1.1, ls=(0, (4, 2)), label='Umbral del canal endémico'),
           Patch(color='#9a9994', alpha=0.35, label='Exceso sobre el umbral'), Patch(color=FONDO, label='Periodo de prueba')]
fig.legend(handles=leyenda, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.0), frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.95)); fig.savefig(SALIDA / 'Figura2_canal_endemico_4_municipios.png', facecolor='white'); plt.show()

## Figura 3. Probabilidad de XGBoost en la prueba

Probabilidad de exceso que XGBoost estimó dos meses antes de cada mes de prueba. La línea discontinua es el umbral de decisión elegido en validación; el fondo gris marca los meses con exceso observado.

In [5]:
fig, axs = plt.subplots(3, 1, figsize=(7.2, 5.6), sharex=True)
for ax, (m, p) in zip(axs, preds.items()):
    u = UMBRAL_XGB[m]
    tramo = (p.exceso != p.exceso.shift()).cumsum()
    for _, g in p[p.exceso == 1].groupby(tramo[p.exceso == 1]):
        ax.axvspan(g.mes_objetivo.iloc[0] - pd.Timedelta(days=15), g.mes_objetivo.iloc[-1] + pd.Timedelta(days=16), color='#e9e8e3', lw=0, zorder=0)
    ax.axhline(u, color=UMB, lw=1.0, ls=(0, (4, 2)), zorder=2)
    ax.plot(p.mes_objetivo, p.prob_XGBoost, color=COL[m], lw=1.8, marker='o', ms=4.5, markeredgecolor='white', markeredgewidth=0.8, zorder=3)
    ax.set_ylim(-0.03, 1.03); ax.yaxis.set_major_locator(MultipleLocator(0.25)); ax.yaxis.set_major_formatter(coma(2))
    ax.set_title(f'{m}  ·  umbral de decisión {u:.2f}'.replace('.', ','), loc='left'); ax.set_ylabel('Prob. de exceso')
    if m == 'Cali':
        ax.text(pd.Timestamp('2025-02-01'), 0.5, 'Prueba de Cali solo hasta dic 2024', fontsize=7.5, color=GRIS, va='center')
axs[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
axs[-1].xaxis.set_major_formatter(FuncFormatter(lambda v, pos: (lambda d: f'{MES[d.month - 1]} {d.year}')(mdates.num2date(v))))
plt.setp(axs[-1].get_xticklabels(), fontsize=7.5)
leyenda = [Line2D([], [], color='#6d6c68', lw=1.8, marker='o', ms=4.5, label='Probabilidad XGBoost (2 meses antes)'),
           Line2D([], [], color=UMB, lw=1.0, ls=(0, (4, 2)), label='Umbral de decisión (validación)'),
           Patch(color='#e9e8e3', label='Mes con exceso observado')]
fig.legend(handles=leyenda, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.0), frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.955)); fig.savefig(SALIDA / 'Figura3_xgboost_prueba.png', facecolor='white'); plt.show()

## Matriz de confusión de XGBoost en la prueba (valores)

Cada mes de prueba se clasifica con el umbral de validación: exceso detectado (VP), exceso no detectado (FN), falsa alarma (FP) y mes normal sin alerta (VN). La sensibilidad y la precisión deben coincidir con `comparacion_modelos_<cod>.csv`.

In [6]:
filas, falsas = [], {}
for m, p in preds.items():
    alerta = (p.prob_XGBoost >= UMBRAL_XGB[m]).astype(int)
    vn, fp, fn, vp = confusion_matrix(p.exceso, alerta, labels=[0, 1]).ravel()
    filas.append({'Municipio': m, 'Meses de prueba': len(p), 'VP': vp, 'FN': fn, 'FP': fp, 'VN': vn,
                  'Sensibilidad': round(vp / (vp + fn), 2), 'Precisión': round(vp / (vp + fp), 2)})
    falsas[m] = ', '.join(f'{MES[t.month - 1]} {t.year}' for t in p.mes_objetivo[(alerta == 1) & (p.exceso == 0)])
tabla2 = pd.DataFrame(filas).set_index('Municipio')
display(tabla2)
tabla2.to_csv(SALIDA / 'matriz_confusion_xgboost.csv')
for m, t in falsas.items():
    print(f'Falsas alarmas {m}: {t}')

,Meses de prueba,VP,FN,FP,VN,Sensibilidad,Precisión
Municipio,,,,,,,
Ibagué,36,18,0,10,8,1.00,0.64
Villavicencio,36,24,1,3,8,0.96,0.89
Cali,24,17,4,1,2,0.81,0.94


Falsas alarmas Ibagué: jun 2023, ene 2025, feb 2025, mar 2025, abr 2025, may 2025, jun 2025, jul 2025, ago 2025, sep 2025
Falsas alarmas Villavicencio: ene 2023, jun 2024, ene 2025
Falsas alarmas Cali: dic 2024


## Figura 4. Matriz de confusión de XGBoost por municipio

La tabla anterior en formato gráfico: un cuadrado por municipio. El número es la cantidad de meses y el color, el porcentaje de cada clase real (por fila), para poder comparar municipios con distinto número de meses de prueba.

In [7]:
fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.75))
for ax, (m, p) in zip(axs, preds.items()):
    alerta = (p.prob_XGBoost >= UMBRAL_XGB[m]).astype(int)
    cm = confusion_matrix(p.exceso.astype(int), alerta, labels=[0, 1]); pct = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(pct, cmap='Blues', vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            c = 'white' if pct[i, j] > 0.55 else '#0b0b0b'
            ax.text(j, i - 0.08, f'{cm[i, j]}', ha='center', va='center', fontsize=13, fontweight='bold', color=c)
            ax.text(j, i + 0.22, f'{pct[i, j] * 100:.0f} %', ha='center', va='center', fontsize=7.5, color=c)
    sens, prec = cm[1, 1] / cm[1].sum(), cm[1, 1] / cm[:, 1].sum()
    ax.set_title(f'{m} ({len(p)} meses)', pad=4)
    ax.set_xlabel(f'Sensibilidad {sens:.2f} · Precisión {prec:.2f}'.replace('.', ','), fontsize=7.8, labelpad=6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Sin alerta', 'Alerta'], fontsize=8); ax.xaxis.tick_top()
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Sin\nexceso', 'Exceso'] if m == 'Ibagué' else ['', ''], fontsize=8)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks([0.5], minor=True); ax.set_yticks([0.5], minor=True)
    ax.grid(False); ax.grid(which='minor', color='white', lw=2); ax.tick_params(which='both', length=0)
axs[0].set_ylabel('Real', fontsize=8.5)
cb = fig.colorbar(im, ax=axs, fraction=0.025, pad=0.02); cb.set_label('% de cada clase real', fontsize=7.5)
cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{v * 100:.0f} %')); cb.ax.tick_params(labelsize=7, length=0); cb.outline.set_visible(False)
fig.savefig(SALIDA / 'Figura4_matriz_confusion_xgboost.png', facecolor='white', bbox_inches='tight'); plt.show()